# Explanation Depth: LIME Comparison, Counterfactuals & Explanation Stability

This notebook implements the **Explanation Depth** phase of the research framework on the **combined 4-site UCI Heart Disease dataset** (N=920, 13-feature schema):
1. **LIME vs. SHAP Local Explainability**: Evaluates top-K feature agreement (Overlap@K) between SHAP (TreeExplainer) and LIME (LimeTabularExplainer) on individual patient predictions.
2. **Counterfactual What-If Reasoning**: Generates minimal realistic feature modifications (e.g., cholesterol, blood pressure, max heart rate, ST depression) to flip high-risk patient predictions below the decision threshold.
3. **Explanation Stability Across CV Folds**: Measures fold-to-fold consistency of SHAP feature rankings across 5 outer cross-validation folds using Overlap@5 and Kendall's tau correlation.


In [1]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Ensure src module is in path
sys.path.append('..')

from src.preprocessing import load_combined_dataset, preprocess_data
from src.models import get_xgboost, get_baseline_models
from src.optimization import optimize_xgboost
from src.ensemble import build_voting_ensemble
from src.lime_explainability import compare_shap_and_lime
from src.counterfactuals import generate_counterfactual_scenarios, MEDICAL_DISCLAIMER
from src.explanation_stability import evaluate_shap_explanation_stability

# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")


Configuration loaded successfully.


## 1. Load Combined Dataset and Fit Models
Load the combined 4-site dataset (920 patients, 13 features) and fit the tuned XGBoost model and soft-voting ensemble.


In [2]:
results_dir = '../' + config['results_dir']
df_combined = load_combined_dataset('../data', results_dir)

X_proc, y, preprocessor, feature_names = preprocess_data(df_combined)

print(f"Loaded Combined Dataset: {df_combined.shape[0]} rows, {X_proc.shape[1]} processed features")

# Fit XGBoost model for SHAP/LIME comparison
xgb_params, _, _ = optimize_xgboost(X_proc, y, cv=config['cv']['inner_folds'], n_trials=config['cv']['optuna_n_trials'], random_state=config['random_state'])
xgb_model = get_xgboost(random_state=config['random_state'], **xgb_params)
xgb_model.fit(X_proc, y)

# Fit soft-voting ensemble
models = get_baseline_models(random_state=config['random_state'])
for m in models.values():
    m.fit(X_proc, y)

ensemble_model = build_voting_ensemble(models, voting='soft')
ensemble_model.fit(X_proc, y)

print("Models trained successfully.")


Loaded Combined Dataset: 920 rows, 13 processed features


Models trained successfully.


## 2. SHAP vs. LIME Explainability Comparison
Compare top-5 local contributing features identified by SHAP (global/local tree explainer) vs. LIME (local linear surrogate) for sample patient predictions.


In [3]:
sample_indices = [0, 1, 2, 5, 8]
df_shap_lime = compare_shap_and_lime(
    xgb_model,
    X_proc,
    feature_names,
    sample_indices=sample_indices,
    output_dir=results_dir,
    top_k=5
)

print("=== SHAP vs. LIME Feature Importance Comparison ===")
display(df_shap_lime)


=== SHAP vs. LIME Feature Importance Comparison ===


,patient_index,top_k,shap_top_features,lime_top_features,overlap_count,overlap_ratio,agreement_status
0,0,5,"cp, oldpeak, exang, sex, age","cp, thal, oldpeak, exang, age",4,80%,Strong Agreement
1,1,5,"cp, exang, thalach, thal, ca","cp, thal, exang, thalach, age",4,80%,Strong Agreement
2,2,5,"oldpeak, cp, exang, thal, ca","cp, thal, oldpeak, exang, age",4,80%,Strong Agreement
3,5,5,"cp, thalach, thal, exang, sex","cp, thal, exang, thalach, ca",4,80%,Strong Agreement
4,8,5,"cp, thal, ca, age, oldpeak","cp, thal, exang, age, ca",4,80%,Strong Agreement


## 3. Counterfactual What-If Reasoning
Generate minimal realistic feature changes required to lower predicted risk for sample high-risk patients.


In [4]:
print("=== " + MEDICAL_DISCLAIMER + " ===\n")

cf_scenarios = generate_counterfactual_scenarios(
    ensemble_model,
    df_combined,
    preprocessor,
    actionable_features=['chol', 'trestbps', 'thalach', 'oldpeak'],
    output_dir=results_dir
)

for idx, sc in enumerate(cf_scenarios):
    print("--- Counterfactual Scenario " + str(idx + 1) + " ---")
    print(sc['statement'])
    print()


=== DISCLAIMER: These counterfactual statements are purely model-based 'what-if' sensitivity analyses computed for explainability purposes. They represent mathematical feature perturbations and do NOT constitute medical advice or clinical guidance. ===



--- Counterfactual Scenario 1 ---
Model-based what-if scenario: If this patient's thalach were modified from 160.0 to 185.0 and oldpeak were modified from 3.6 to 1.0, with other factors unchanged, the model's predicted risk would drop from 79.0% to 49.9% (below the 50% threshold).

--- Counterfactual Scenario 2 ---
Model-based what-if scenario: If this patient's thalach were modified from 155.0 to 170.7 and oldpeak were modified from 3.1 to 0.4, with other factors unchanged, the model's predicted risk would drop from 80.3% to 50.0% (below the 50% threshold).

--- Counterfactual Scenario 3 ---
No simple counterfactual modification in ['chol', 'trestbps', 'thalach', 'oldpeak'] was sufficient to flip prediction below 50%.

--- Counterfactual Scenario 4 ---
Model-based what-if scenario: If this patient's chol were modified from 229.0 to 226.4, with other factors unchanged, the model's predicted risk would drop from 51.6% to 43.4% (below the 50% threshold).



## 4. SHAP Explanation Stability Across CV Folds
Evaluate fold-to-fold ranking stability of top SHAP features across 5 outer cross-validation folds using Overlap@5 and Kendall's tau correlation.


In [5]:
print("Evaluating SHAP explanation stability across 5 outer CV folds...")
stability_results = evaluate_shap_explanation_stability(
    df_combined,
    target_col='target',
    outer_splits=config['cv']['outer_folds'],
    top_k=5,
    random_state=config['random_state']
)

print(f"\nMean Overlap@5: {stability_results['mean_overlap_at_k']:.1%}")
print(f"Mean Kendall's Tau: {stability_results['mean_kendall_tau']:.3f}")

print("\n=== Top-5 Features per Fold ===")
for fold_idx, top_feats in enumerate(stability_results['fold_top_k_features']):
    print("  Fold " + str(fold_idx + 1) + ": " + ", ".join(top_feats))

print("\n=== Interpretation ===\n" + stability_results['interpretation'])


Evaluating SHAP explanation stability across 5 outer CV folds...



Mean Overlap@5: 70.0%
Mean Kendall's Tau: 0.723

=== Top-5 Features per Fold ===
  Fold 1: cp, oldpeak, chol, sex, age
  Fold 2: cp, oldpeak, exang, age, thal
  Fold 3: cp, oldpeak, chol, thalach, exang
  Fold 4: cp, oldpeak, exang, sex, chol
  Fold 5: cp, exang, oldpeak, sex, age

=== Interpretation ===
Moderate Explanation Stability: Top-5 SHAP features exhibit an average Overlap@5 of 70.0% across folds (Kendall's tau: 0.723). Core risk drivers remain largely stable, with minor re-ordering in secondary features across folds.
